# LoRA demo: Hugging Face PEFT + tiny GPT-2

Transfer Learning, кастомізація LLM і RAG

**Мета:** показати мінімальний pipeline parameter-efficient fine-tuning через LoRA.

У цьому notebook ми:

1. завантажимо маленьку causal language model;
2. подивимося, скільки параметрів має base model;
3. додамо LoRA adapters через `peft`;
4. навчимо тільки adapter-параметри на маленькому synthetic dataset;
5. порівняємо генерацію до і після LoRA;
6. збережемо adapter окремо від base model.

За замовчуванням використовується `sshleifer/tiny-gpt2`, бо він швидко запускається навіть на CPU. Для ближчої до реального прикладу демонстрації можна змінити `MODEL_NAME` на `distilbert/distilgpt2`.

## Встановлення бібліотек

In [ ]:
# !pip -q install "torch>=2.2" "transformers>=4.40" "peft>=0.18" "accelerate>=0.28"


## 1. Імпорти та налаштування

`RUN_TRAINING = True` запускає коротке навчання adapter. Якщо потрібно тільки показати структуру LoRA без тренування, змініть на `False`.


In [ ]:

from pathlib import Path
import random

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# CPU-friendly default. For a more realistic demo try: "distilbert/distilgpt2".
MODEL_NAME = "sshleifer/tiny-gpt2"
# MODEL_NAME = "distilbert/distilgpt2"

OUTPUT_DIR = Path("models/lora_gpt2_support_adapter")
RUN_TRAINING = True
NUM_EPOCHS = 8
BATCH_SIZE = 4
LEARNING_RATE = 5e-4
MAX_LENGTH = 160

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 2. Завантажуємо tokenizer і base model

GPT-2-подібні моделі не мають окремого `pad_token`, тому для batch training зазвичай ставлять `pad_token = eos_token`.


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.to(DEVICE)

print("Model:", MODEL_NAME)
print("Pad token:", repr(tokenizer.pad_token), "id:", tokenizer.pad_token_id)


## 3. Допоміжні функції

Порахуємо параметри і зробимо функцію для короткої генерації. Це допоможе порівняти base model і LoRA model.


In [ ]:

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def print_parameter_report(model, title):
    total, trainable = count_parameters(model)
    pct = 100 * trainable / total
    print(title)
    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,} ({pct:.4f}%)")


def build_prompt(user_text):
    return (
        "### Запит клієнта:\n"
        f"{user_text}\n\n"
        "### Відповідь асистента:\n"
    )


def generate_text(model, user_text, max_new_tokens=70):
    model.eval()
    prompt = build_prompt(user_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


print_parameter_report(base_model, "Base model before LoRA")


## 4. Перевіряємо baseline-відповідь

До fine-tuning модель не знає наш формат відповіді. На tiny model текст може бути майже випадковим, і це нормально для демонстрації.


In [ ]:

test_request = "Клієнт пише, що оплатив тариф, але підписка не активувалася."
print(generate_text(base_model, test_request))


## 5. Знаходимо GPT-2 modules для LoRA

У GPT-2 основні projection layers мають назви на кшталт `c_attn` і `c_proj`. Саме до них часто додають LoRA adapters.


In [ ]:

for name, module in base_model.named_modules():
    if any(part in name for part in ["c_attn", "c_proj"]):
        print(f"{name:<45} {module.__class__.__name__}")


## 6. Додаємо LoRA adapters через PEFT

LoRA заморожує base model і додає маленькі trainable матриці до вибраних шарів. Тому ми навчаємо лише невелику частку параметрів.

Ключові параметри:

- `r` - rank adapter-матриць;
- `lora_alpha` - масштаб LoRA-оновлення;
- `target_modules` - куди саме вставити adapters;
- `task_type="CAUSAL_LM"` - тип задачі для autoregressive language model.


In [ ]:

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["c_attn", "c_proj"],
    fan_in_fan_out=True,
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.to(DEVICE)

print_parameter_report(lora_model, "Model after adding LoRA")
lora_model.print_trainable_parameters()


## 7. Малий instruction dataset

Задача: навчити модель відповідати у простому форматі для support-router:

- визначити категорію звернення;
- дати коротку відповідь клієнту;
- запропонувати наступну дію.


In [ ]:

training_examples = [
    {
        "user": "Клієнт не може увійти в акаунт, бо SMS-код не приходить.",
        "assistant": "Категорія: TECHNICAL\nВідповідь: Перевірте номер телефону та повторіть запит коду через кілька хвилин. Якщо код не прийде, передамо звернення технічній підтримці.\nНаступна дія: перевірити доставку SMS.",
    },
    {
        "user": "Після оновлення додаток зависає на екрані входу.",
        "assistant": "Категорія: TECHNICAL\nВідповідь: Спробуйте очистити кеш додатка або перевстановити його. Якщо проблема залишиться, нам потрібна модель пристрою та версія ОС.\nНаступна дія: зібрати технічні деталі.",
    },
    {
        "user": "Користувач бачить помилку 500 під час відкриття звіту.",
        "assistant": "Категорія: TECHNICAL\nВідповідь: Це схоже на серверну помилку. Ми перевіримо журнал подій і повідомимо, коли звіт знову буде доступний.\nНаступна дія: створити технічний ticket.",
    },
    {
        "user": "З картки двічі списали оплату за один тарифний план.",
        "assistant": "Категорія: BILLING\nВідповідь: Перевіримо дубльоване списання і повернемо зайву оплату, якщо транзакція підтвердиться.\nНаступна дія: запросити дату, суму та останні 4 цифри картки.",
    },
    {
        "user": "Оплата пройшла, але підписка не активувалася.",
        "assistant": "Категорія: BILLING\nВідповідь: Дякуємо за повідомлення. Перевіримо статус платежу і вручну активуємо підписку, якщо оплата успішна.\nНаступна дія: перевірити payment id або чек.",
    },
    {
        "user": "Клієнт просить рахунок для бухгалтерії за минулий місяць.",
        "assistant": "Категорія: BILLING\nВідповідь: Ми можемо підготувати рахунок за потрібний період і надіслати його на email акаунта.\nНаступна дія: уточнити юридичні реквізити.",
    },
    {
        "user": "Замовлення не доставили у вказаний день.",
        "assistant": "Категорія: DELIVERY\nВідповідь: Перевіримо статус доставки у перевізника і повідомимо новий очікуваний час прибуття.\nНаступна дія: уточнити номер замовлення.",
    },
    {
        "user": "Клієнт хоче змінити адресу доставки після оформлення замовлення.",
        "assistant": "Категорія: DELIVERY\nВідповідь: Якщо замовлення ще не передано перевізнику, ми змінимо адресу доставки.\nНаступна дія: перевірити статус відправлення.",
    },
    {
        "user": "Посилка має статус delivered, але клієнт її не отримав.",
        "assistant": "Категорія: DELIVERY\nВідповідь: Перевіримо підтвердження доставки і зв'яжемося з перевізником для уточнення деталей.\nНаступна дія: відкрити delivery investigation.",
    },
]

print("Examples:", len(training_examples))
print(training_examples[0]["user"])
print(training_examples[0]["assistant"])


## 8. Dataset для causal language modeling

Ми навчаємо модель продовжувати prompt правильною відповіддю. Для простоти loss рахується на всій послідовності. У production SFT часто маскують prompt-токени і рахують loss тільки на відповіді assistant.


In [ ]:

class SupportDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.rows = []
        for item in examples:
            text = build_prompt(item["user"]) + item["assistant"] + tokenizer.eos_token
            encoded = tokenizer(
                text,
                max_length=max_length,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            input_ids = encoded["input_ids"].squeeze(0)
            attention_mask = encoded["attention_mask"].squeeze(0)
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            self.rows.append({
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
            })

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


dataset = SupportDataset(training_examples, tokenizer, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

batch = next(iter(dataloader))
print({key: value.shape for key, value in batch.items()})


## 9. Коротке тренування LoRA

Навчаються тільки adapter-параметри. На CPU це все одно може зайняти трохи часу для `distilgpt2`, тому для заняття краще спершу запускати `sshleifer/tiny-gpt2`.


In [ ]:

if RUN_TRAINING:
    trainable_params = [p for p in lora_model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)
    lora_model.train()

    step = 0
    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0.0
        for batch in dataloader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            outputs = lora_model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            step += 1

        mean_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | loss: {mean_loss:.4f}")
else:
    print("RUN_TRAINING = False, training skipped.")


## 10. Генерація після LoRA

На такому маленькому dataset результат не має бути ідеальним. Для демонстрації важливо, що модель починає наслідувати формат: `Категорія`, `Відповідь`, `Наступна дія`.


In [ ]:

for request in [
    "Клієнт пише, що оплатив тариф, але підписка не активувалася.",
    "Клієнт не отримав посилку, хоча статус показує delivered.",
    "Після входу в кабінет сторінка звітів відкривається з помилкою 500.",
]:
    print("=" * 90)
    print(generate_text(lora_model, request))


## 11. Зберігаємо LoRA adapter

PEFT зберігає adapter окремо від base model. Це зручно: base model може залишатися спільною, а adapter-файли маленькі і відповідають конкретній задачі або стилю.


In [ ]:

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
lora_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved adapter to:", OUTPUT_DIR.resolve())
print("Files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)


## 12. Завантажуємо adapter назад

Для inference потрібно завантажити ту саму base model і накласти на неї adapter через `PeftModel.from_pretrained`.


In [ ]:

base_for_inference = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_for_inference.config.pad_token_id = tokenizer.pad_token_id

loaded_lora_model = PeftModel.from_pretrained(base_for_inference, OUTPUT_DIR)
loaded_lora_model.to(DEVICE)

print(generate_text(loaded_lora_model, "Клієнт просить рахунок для бухгалтерії за минулий місяць."))


## 13. Optional: merge adapter into base model

Для deployment іноді adapter зливають із base model. Це робить одну звичайну model checkpoint, але втрачає гнучкість швидко перемикати adapters.


In [ ]:

# merged_model = loaded_lora_model.merge_and_unload()
# merged_model.save_pretrained("models/lora_gpt2_support_merged")
# tokenizer.save_pretrained("models/lora_gpt2_support_merged")


## Довідка

Офіційні матеріали, які корисно відкрити після demo:

- [Hugging Face PEFT: LoRA package reference](https://huggingface.co/docs/peft/package_reference/lora)
- [Transformers: Parameter-efficient fine-tuning](https://huggingface.co/docs/transformers/peft)
- [PEFT model API](https://huggingface.co/docs/peft/package_reference/peft_model)


## Ключова ідея

Full fine-tuning змінює всі ваги моделі. LoRA залишає base model замороженою і навчає маленькі adapter-матриці у вибраних шарах. Тому adapter дешевше тренувати, зберігати і переносити між середовищами.
